## Imports

In [1]:
from transformers import T5Tokenizer, T5ForConditionalGeneration
import google.generativeai as genai
from jinja2 import Template
from copy import deepcopy
from tqdm import tqdm
import pandas as pd
import dotenv
import torch
import json
import os
import re

dotenv.load_dotenv()
genai.configure(api_key=os.getenv("GEMINI_API_KEY"))
model = genai.GenerativeModel("gemini-2.5-flash-lite")

/home/mafr2/LaBo/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/mafr2/LaBo/.venv/lib/python3.9/site-packages/google/api_core/_python_version_support.py:246: FutureWarning: You are using a non-supported Python version (3.9.25). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.api_core.
  warnings.warn(message, FutureWarning)
/home/mafr2/LaBo/.venv/lib/python3.9/site-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  w

In [8]:
with open("all_datasets_classes.json", "r") as f:
    all_datasets_classes = json.load(f)

In [9]:
selected_datasets = ["CIFAR10", "CUB", "flower"]
dataset_classes = {dataset: all_datasets_classes[dataset] for dataset in selected_datasets}

In [25]:
base_prompt=Template(
"""Use 10 sentences to describe the {{ prompt }}:

Your ouput should be in the following format:
<sentence>sentence 1</sentence>
<sentence>sentence 2</sentence>
...
<sentence>sentence 10</sentence>""")

prompts = [
    """what the {{ class_name }} looks like""",
    """the appearance of the {{ class_name }}""",
    """the color of the {{ class_name }}""",
    """the pattern of the {{ class_name }}""",
    """the shape of the {{ class_name }}""",
]

prompt_templates = [Template(prompt) for prompt in prompts]

In [5]:
def prepare_model_prompts(dataset_classes, prompt_templates):
    model_prompts = {}
    classes_modified = {}

    # Adding superclass information for flower datasets
    for dataset, classes in dataset_classes.items():
        if dataset == "flower":
            classes_modified[dataset] = [class_name + ' flower' for class_name in classes]
        else:
            classes_modified[dataset] = classes

    for dataset, classes in classes_modified.items():
        model_prompts[dataset] = {}
        for class_name in classes:
            model_prompts[dataset][class_name] = []
            for prompt_template in prompt_templates:
                prompt = base_prompt.render(prompt=prompt_template.render(class_name=class_name))
                model_prompts[dataset][class_name].append(prompt)
                
    return model_prompts

In [6]:
def generate_content(prompt):
    response = model.generate_content(
        prompt,
        generation_config={
            "temperature": 0.9,
            "top_p": 1.0,
            "max_output_tokens": 1000
        }
    )
    return response.text

In [7]:
def generate_prompts(model_prompts, run=1):
    pd_frames = []
    for dataset, classes in model_prompts.items():
        dataset_results = []                          # fix 2: per-dataset list
        for class_name, prompts in classes.items():
            for prompt in prompts:
                row = {
                    "dataset": dataset,
                    "class_name": class_name,
                    "prompt": prompt,
                    "content": ''
                }
                dataset_results.append(row)

        file_path = f"llm_outputs/{dataset}/run_{run}.csv"  # fix 1: define before use
        os.makedirs(f"llm_outputs/{dataset}", exist_ok=True)
        df = pd.DataFrame(dataset_results)            # fix 3: store df, call to_csv separately
        df.to_csv(file_path, index=False)
        pd_frames.append((file_path, df))

    return pd_frames

def run_generation(file_path):
    results = []
    df = pd.read_csv(file_path)
    for prompt in tqdm(df["prompt"], desc=f"Generating content {file_path.split('/')[-1]}"):
        content = generate_content(prompt)
        results.append(content)
    
    df["content"] = results
    df.to_csv(file_path, index=False)
    return results

In [29]:
model_prompts = prepare_model_prompts(dataset_classes, prompt_templates)

# total number of prompts
total_prompts = sum(len(prompts) for dataset in model_prompts.values() for prompts in dataset.values())
print(f"Total number of prompts: {total_prompts * 10}")

Total number of prompts: 5100


In [64]:
for run in range(1, 2):
    pd_frames = generate_prompts(model_prompts, run=run)
    for file_path, _ in pd_frames:
        run_generation(file_path)

Generating content run_1.csv: 100%|██████████| 50/50 [01:48<00:00,  2.17s/it]


In [2]:
from transformers import T5Tokenizer, T5ForConditionalGeneration
import torch

device = "mps" if torch.backends.mps.is_available() else "cuda:0" if torch.cuda.is_available() else "cpu"

t5_tokenizer = T5Tokenizer.from_pretrained("t5-large")

t5_model = T5ForConditionalGeneration.from_pretrained(
    "t5-large-concept-extractor"
).to(device)

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


In [3]:
def sentences2concepts(sentences, batch_size=16):

    task_prefix = "extract concepts from sentence: "

    concepts = []
    outputs = []

    for i in range(0, len(sentences), batch_size):

        batch = sentences[i:i + batch_size]

        inputs = t5_tokenizer(
            [task_prefix + s for s in batch],
            return_tensors="pt",
            padding=True,
            truncation=True
        )

        input_ids = inputs["input_ids"].to(device)
        attention_mask = inputs["attention_mask"].to(device)

        output_sequences = t5_model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_length=64
        )

        decoded = t5_tokenizer.batch_decode(
            output_sequences,
            skip_special_tokens=True
        )

        outputs.extend(decoded)

    for out in outputs:
        concepts.extend(out.split("; "))

    return concepts

In [4]:
def extract_sentences(text: str) -> list[str]:
    return re.findall(r"<sentence>(.*?)</sentence>", text, re.DOTALL)

In [16]:
for dataset in os.listdir("llm_out/"):
    for run in range(1, 11):
        file_path = os.path.join("llm_out", dataset, f"run_{run}.csv")
        df = pd.read_csv(file_path)
        extracted_sentences = [extract_sentences(str(content)) for content in df["content"].tolist()]

        df["sentence_count"] = [len(s) for s in extracted_sentences]
        under_10_idx = df[df["sentence_count"] == 0].index.tolist()

        if not under_10_idx:
            continue

        print(f"\n{file_path} — regenerating {len(under_10_idx)} row(s) with < 10 sentences...")

        for idx in tqdm(under_10_idx, desc=f"Re-running {os.path.basename(file_path)}"):
            prompt = df.at[idx, "prompt"]
            # new_content = generate_content(prompt)
            # new_sentences = re.findall(r"<concept>(.*?)</concept>", new_content, re.DOTALL)
            # df.at[idx, "content"] = new_content
            # df.at[idx, "sentence_count"] = len(new_sentences)
            print(f"  [{df.at[idx, 'class_name']}]")# {len(new_sentences)} sentences after retry")

        df.to_csv(file_path, index=False)
        print(f"Saved {file_path}")


In [17]:
for dataset in os.listdir('llm_out'):
    records = []
    for i in range(1, 11):
        file_path = os.path.join('llm_out', dataset, f"run_{i}.csv")
        df = pd.read_csv(file_path)
        extracted_sentences = [extract_sentences(str(content)) for content in df["content"].tolist()]
        
        # max_sentences = max((len(s) for s in extracted_sentences), default=0)
        # for i in range(max_sentences):
        #     df[f"sentence_{i+1}"] = [s[i] if i < len(s) else None for s in extracted_sentences]

        df['run'] = [i] * len(df)
        df = df[['run','dataset', 'class_name', 'prompt', 'content']]  # reorder columns

        records.append(df)
    
    combined_df = pd.concat(records, ignore_index=True)
    combined_df.to_csv(os.path.join('llm_out', dataset, "combined_results.csv"), index=False)

In [23]:
for dataset in os.listdir('llm_out'):
    data_json = {dataset: {}}
    df = pd.read_csv(file_path)
    for class_name in tqdm(df['class_name'].unique(), desc=f"Processing {dataset}"):
        data_json[dataset][class_name] = {"sentences": []}
        extracted_sentences = [extract_sentences(str(content)) for content in df["content"].tolist()]
        data_json[dataset][class_name]["sentences"] = [s for sublist in extracted_sentences for s in sublist]  # flatten list of lists
        data_json[dataset][class_name]["concepts"] = sentences2concepts(data_json[dataset][class_name]["sentences"])

    # dump json file
    with open(os.path.join('llm_out', dataset, "data.json"), "w") as f:
        json.dump(data_json, f, indent=4)

Processing CIFAR10: 100%|██████████| 10/10 [03:53<00:00, 23.39s/it]


In [22]:
df = pd.read_csv("llm_out/flower/combined_results.csv")
df['class_name'] = df['class_name'] + '.'
df['class_name'] = df['class_name'].str.replace(' flower.', '', regex=False)
df.to_csv("llm_out/flower/combined_results.csv", index=False)

In [23]:
df = pd.read_csv("llm_out/flower/combined_results.csv")
df['class_name'].value_counts()

class_name
blackberry lily    50
passion flower     50
water lily         50
cyclamen           50
watercress         50
                   ..
clematis           50
hibiscus           50
lotus              50
anthurium          50
thorn apple        50
Name: count, Length: 102, dtype: int64

In [12]:
import re
from typing import List, Dict, Any


def normalize_text(text: str) -> str:
    """Lowercase and collapse whitespace."""
    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text


def tokenize(text: str) -> List[str]:
    """Tokenize into word tokens."""
    return re.findall(r"\b\w+\b", normalize_text(text))


def clean_concept_list(
    concepts: List[str],
    class_names: List[str],
    super_class: str = "object",
    verbose: bool = True,
) -> List[str]:
    """
    Clean concepts using LaBo Appendix B.4 heuristics:

    1. If an exact class name appears in the concept, replace it with super_class.
    2. For multi-token class names, if all class-name tokens appear in the concept
       (possibly reordered), delete the concept — unless that class was already
       handled by heuristic 1.
    3. Remove the word 'overall' from concepts.

    Returns cleaned concepts.
    """
    cleaned = []
    replaced_count = 0
    deleted_count = 0

    super_class_norm = normalize_text(super_class)
    class_names_norm = [normalize_text(c) for c in class_names]

    for concept in concepts:
        original_concept = concept
        concept = normalize_text(concept)
        concept = re.sub(r"\boverall, \b", "", concept)
        concept = re.sub(r"\s+", " ", concept).strip()
        if not concept:
            continue

        replaced_classes = set()

        # Heuristic 1: exact phrase replacement
        for class_name_norm in class_names_norm:
            pattern = r"\b" + re.escape(class_name_norm) + r"\b"

            if re.search(pattern, concept):
                new_concept = re.sub(pattern, super_class_norm, concept)
                if new_concept != concept:
                    concept = re.sub(r"\s+", " ", new_concept).strip()
                    replaced_classes.add(class_name_norm)
                    replaced_count += 1
                    if verbose:
                        print(f"  [REPLACED] '{original_concept}' -> '{concept}'")

        # Heuristic 2: delete if all tokens of a multi-token class name are present
        concept_tokens = set(tokenize(concept))
        keep_concept = True

        for class_name_norm in class_names_norm:
            if class_name_norm in replaced_classes:
                continue

            class_tokens = tokenize(class_name_norm)
            if len(class_tokens) <= 1:
                continue

            if all(tok in concept_tokens for tok in class_tokens):
                keep_concept = False
                deleted_count += 1
                if verbose:
                    print(f"  [DELETED]  '{original_concept}' (contains all tokens of '{class_name_norm}')")
                break

        # Additional rule: if concept collapses to only the super class, drop it
        if keep_concept and concept == super_class_norm:
            keep_concept = False
            deleted_count += 1
            if verbose:
                print(f"  [DELETED]  '{original_concept}' (collapsed to super class '{super_class_norm}')")

        if keep_concept and concept:
            cleaned.append(concept)

    if verbose:
        print("\n--- Summary ---")
        print(f"Original concepts: {len(concepts)}")
        print(f"Replaced: {replaced_count}")
        print(f"Deleted: {deleted_count}")
        print(f"Cleaned concepts: {len(cleaned)}")

    return cleaned


def clean_dataset_one_class_at_a_time(
    data_json: Dict[str, Dict[str, Any]],
    dataset_key: str,
    super_class: str = "object",
    verbose: bool = False,
) -> Dict[str, Dict[str, Any]]:
    """
    Clean concepts class-by-class while keeping the same structure:
    data_json[dataset_key][class_name]["extracted_concepts"]
    """
    class_names = list(data_json[dataset_key].keys())

    for class_name, payload in data_json[dataset_key].items():
        raw_concepts = payload.get("concepts", [])
        if verbose:
            print(f"\n[{dataset_key}/{class_name}] cleaning {len(raw_concepts)} concepts")

        payload["extracted_concepts"] = clean_concept_list(
            concepts=raw_concepts,
            class_names=class_names,
            super_class=super_class,
            verbose=verbose,
        )

    return data_json

In [ ]:
with open('llm_out/CUB/data.json', 'r') as f:
    data = json.load(f)

super_class = 'bird'

# clean per class and keep the same nested structure
data = clean_dataset_one_class_at_a_time(
    data_json=data,
    dataset_key='CUB',
    super_class=super_class,
    verbose=True,
)

to_write = {}
for classes in data['CUB'].keys():
    to_write[classes] = data['CUB'][classes]['extracted_concepts']

with open('datasets/CUB/concepts/class2concepts_gemini.json', 'w') as f:
    json.dump(to_write, f, indent=4)


[CUB/black footed albatross] cleaning 775 concepts
  [DELETED]  'hallmark of the black-footed albatross' (contains all tokens of 'black footed albatross')

--- Summary ---
Original concepts: 775
Replaced: 0
Deleted: 1
Cleaned concepts: 774

[CUB/laysan albatross] cleaning 812 concepts
  [REPLACED] 'when in flight, the Laysan Albatross appears graceful and streamlined' -> 'when in flight, the bird appears graceful and streamlined'

--- Summary ---
Original concepts: 812
Replaced: 1
Deleted: 0
Cleaned concepts: 812

[CUB/sooty albatross] cleaning 798 concepts
  [REPLACED] 'in essence, the sooty albatross is a master of the air' -> 'in essence, the bird is a master of the air'

--- Summary ---
Original concepts: 798
Replaced: 1
Deleted: 0
Cleaned concepts: 798

[CUB/groove billed ani] cleaning 800 concepts
  [DELETED]  'groove-billed ani displays pattern of social cooperation, communal nesting, and opportunistic foraging' (contains all tokens of 'groove billed ani')
  [DELETED]  'combina

: 

In [5]:
device = "mps" if torch.backends.mps.is_available() else "cuda:0" if torch.cuda.is_available() else "cpu"

t5_tokenizer = T5Tokenizer.from_pretrained("t5-large")

t5_model = T5ForConditionalGeneration.from_pretrained(
    "t5-large-concept-extractor"
).to(device)

In [13]:
import pandas as pd

df = pd.read_csv('llm_out/CIFAR10/combined_results.csv')
df.head()

,run,dataset,class_name,prompt,content
0,1,CIFAR10,airplane,Use 10 sentences to describe the what the airp...,<sentence>The airplane gleamed under the midda...
1,1,CIFAR10,airplane,Use 10 sentences to describe the the appearanc...,"<sentence>The aircraft was a sleek, silvery bi..."
2,1,CIFAR10,airplane,Use 10 sentences to describe the the color of ...,<sentence>The airplane gleamed under the midda...
3,1,CIFAR10,airplane,Use 10 sentences to describe the the pattern o...,<sentence>The airplane's pattern is a series o...
4,1,CIFAR10,airplane,Use 10 sentences to describe the the shape of ...,"<sentence>The airplane possesses a sleek, aero..."


In [14]:
for run in df['run'].unique():
    df[df['run'] == run][['dataset','class_name', 'prompt', 'content']].to_csv(f'llm_out/CIFAR10/run_{run}.csv', index=False)

In [6]:
def sentences2concepts(sentences, class_name, batch_size=128):

    task_prefix = "extract concepts from sentence: "

    concepts = []
    outputs = []

    for i in range(0, len(sentences), batch_size):

        batch = sentences[i:i + batch_size]

        inputs = t5_tokenizer(
            [task_prefix + s for s in batch],
            return_tensors="pt",
            padding=True,
            truncation=True
        )

        input_ids = inputs["input_ids"].to(device)
        attention_mask = inputs["attention_mask"].to(device)

        with torch.no_grad():
            output_sequences = t5_model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_length=64,
                num_beams=4,
                early_stopping=True
            )

        decoded = t5_tokenizer.batch_decode(
            output_sequences,
            skip_special_tokens=True
        )

        outputs.extend(decoded)

    for out in outputs:
        concepts.extend(out.split("; "))

    return concepts

def extract_sentences(text: str) -> list[str]:
    return re.findall(r"<sentence>(.*?)</sentence>", text, re.DOTALL)

In [10]:
import json
import pandas as pd
import re
from tqdm import tqdm

with open('llm_out/CUB/data.json', 'r') as f:
    data_json = json.load(f)

df = pd.read_csv("llm_out/CUB/combined_results.csv")

for classes in tqdm(list(set(dataset_classes['CUB']) - set(data_json['CUB'].keys())), desc="Processing CUB classes"):
    data_json['CUB'][classes] = {"sentences": []}
    extracted_sentences = [extract_sentences(str(content)) for content in df[df['class_name'] == classes]["content"].tolist()]
    data_json['CUB'][classes]["sentences"] = [s for sublist in extracted_sentences for s in sublist]  # flatten list of lists
    data_json['CUB'][classes]["concepts"] = sentences2concepts(data_json['CUB'][classes]["sentences"], 'CUB')

    # dump json file
    with open(os.path.join('llm_out', 'CUB', "data.json"), "w") as f:
        json.dump(data_json, f, indent=4)

Processing CUB classes: 100%|██████████| 30/30 [15:49<00:00, 31.67s/it]


In [10]:
list(set(dataset_classes['CUB']) - set(data['CUB'].keys()))

['prairie warbler',
 'cactus wren',
 'wilson warbler',
 'warbling vireo',
 'american three toed woodpecker',
 'canada warbler',
 'swainson warbler',
 'pine warbler',
 'nashville warbler',
 'rock wren',
 'chestnut sided warbler',
 'kentucky warbler',
 'red headed woodpecker',
 'red eyed vireo',
 'red bellied woodpecker',
 'tennessee warbler',
 'white eyed vireo',
 'cape may warbler',
 'cedar waxwing',
 'black throated blue warbler',
 'blue winged warbler',
 'bohemian waxwing',
 'myrtle warbler',
 'northern waterthrush',
 'prothonotary warbler',
 'louisiana waterthrush',
 'worm eating warbler',
 'golden winged warbler',
 'bewick wren',
 'common yellowthroat',
 'yellow warbler',
 'palm warbler',
 'magnolia warbler',
 'downy woodpecker',
 'yellow throated vireo',
 'red cockaded woodpecker',
 'cerulean warbler',
 'black and white warbler',
 'winter wren',
 'carolina wren',
 'orange crowned warbler',
 'bay breasted warbler',
 'pileated woodpecker',
 'mourning warbler',
 'house wren',
 'hoode

In [3]:
len(data['CUB'].keys())

153

In [27]:
to_write['bird']

['sleek, iridescent body',
 'shimmered with shades of emerald and sapphire',
 'head was adorned with crest of fine, almost feathery plumes',
 'tipped with a vibrant crimson',
 'large, intelligent eyes, the color of polished obsidian',
 'the color of polished obsidian',
 'sharply curved beak',
 'deep ebony, suggested a specialized diet',
 'wings were broad and powerful',
 'primary feathers edged in a contrasting band of pure white',
 'feathers cascaded downwards, displaying a subtle gradient of blues and greens',
 'legs were thin and wiry',
 'delicate, dark claws adapted for gripping branches',
 'small patch of bright yellow marked the skin around its eyes',
 'add a startling splash of color',
 'underside of body was a soft, creamy white',
 'stark contrast to its jewel-toned back',
 'it presented a picture of exquisite beauty and intricate detail',
 'dazzling spectacle of iridescent blues and greens',
 'feathers shimmered like scattered jewels in the sunlight',
 'vibrant tapestry of col

In [ ]:
def normalize_text(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text


def tokenize(text: str):
    return re.findall(r"\b\w+\b", text.lower())


def remove_class_name_from_concept(concept: str, class_names, super_class: str):
    # Strip parentheses first so e.g. "(airplane)" is treated as "airplane"
    result = normalize_text(concept)
    result = re.sub(r"[()]", "", result)
    result = re.sub(r"\s+", " ", result).strip()

    concept_tokens = set(tokenize(result))

    for class_name in class_names:
        class_norm = normalize_text(class_name)
        class_tokens = tokenize(class_norm)

        # Heuristic 2: all tokens present for multi-token class name -> delete
        if len(class_tokens) > 1 and all(tok in concept_tokens for tok in class_tokens):
            return None

        # Heuristic 1: exact phrase match -> replace with super-class (applied to all class names)
        pattern = r"\b" + re.escape(class_norm) + r"\b"
        if re.search(pattern, result):
            result = re.sub(pattern, super_class, result)
            result = re.sub(r"\s+", " ", result).strip()

    # Delete if the concept is just the super-class word alone (no meaningful content left)
    if result.strip() == super_class.strip():
        return None

    # Heuristic 3: strip leading "overall," or "overall " — too generic as a prefix
    result = re.sub(r"^overall[,\s]+", "", result).strip()

    return result if result else None


def clean_concepts(concepts, class_names, super_class):
    cleaned = []
    for c in concepts:
        out = remove_class_name_from_concept(c, class_names, super_class)
        if out is not None and out != "":
            cleaned.append(out)

    return list(dict.fromkeys(cleaned))

In [126]:
# Super-class name used as replacement when a class name is found in a concept
DATASET_SUPER_CLASS = {
    "aircraft":  "aircraft",
    "CIFAR10":   "object",
    "CIFAR100":  "object",
    "CUB":       "bird",
    "DTD":       "texture",
    "flower":    "flower",
    "food":      "food",
    "HAM10000":  "skin lesion",
    "ImageNet":  "object",
    "RESISC45":  "scene",
    "UCF101":    "action",
}

for dataset in os.listdir('llm_outputs'):
    file_path = os.path.join('llm_outputs', dataset, "data.json")
    with open(file_path, "r") as f:
        data_json = json.load(f)

    super_class = DATASET_SUPER_CLASS.get(dataset, "object")
    all_class_names = list(data_json[dataset].keys())

    for class_name, content in tqdm(data_json[dataset].items(), desc=f"Cleaning concepts [{dataset}]"):
        raw_concepts = content["concepts"]
        cleaned = clean_concepts(raw_concepts, all_class_names, super_class)

        # Print any changes made
        for orig in raw_concepts:
            out = remove_class_name_from_concept(orig, all_class_names, super_class)
            if out is None or out.strip() == "":
                print(f"  [DELETED]  '{orig}'")
            elif out != normalize_text(orig):
                print(f"  [REPLACED] '{orig}' -> '{out}'")

        data_json[dataset][class_name]["extracted_concepts"] = cleaned

    with open(file_path, "w") as f:
        json.dump(data_json, f, indent=4)
    print(f"[{dataset}] Saved cleaned concepts to {file_path}")
    # print(f"[{dataset}] Dry run complete — save is commented out")


Cleaning concepts [CIFAR10]:  40%|████      | 4/10 [00:00<00:00, 32.84it/s]

  [REPLACED] 'overall impression was one of robust engineering and advanced design' -> 'impression was one of robust engineering and advanced design'
  [REPLACED] 'even from a distance, the airplane exuded an aura of power and sophistication' -> 'even from a distance, the object exuded an aura of power and sophistication'
  [REPLACED] 'overall, the airplane presented an image of advanced engineering and quiet power' -> 'the object presented an image of advanced engineering and quiet power'
  [REPLACED] 'powerful red hue gave the airplane a dynamic and almost fiery presence as it sat on the tarmac' -> 'powerful red hue gave the object a dynamic and almost fiery presence as it sat on the tarmac'
  [DELETED]  '(airplane)'
  [REPLACED] 'overall, the design is modern and eye-catching' -> 'the design is modern and eye-catching'
  [REPLACED] 'suggests a sleek and fast identity for the airplane' -> 'suggests a sleek and fast identity for the object'
  [REPLACED] 'overall, the airplane exhibits

Cleaning concepts [CIFAR10]: 100%|██████████| 10/10 [00:00<00:00, 33.04it/s]

  [REPLACED] 'overall, the horse presents a picture of elegant power and quiet dignity' -> 'the object presents a picture of elegant power and quiet dignity'
  [REPLACED] 'overall impression was one of warmth, strength, and a timeless elegance' -> 'impression was one of warmth, strength, and a timeless elegance'
  [REPLACED] 'most striking feature of the horse's pattern is its coat color' -> 'most striking feature of the object's pattern is its coat color'
  [DELETED]  '(horse)'
  [REPLACED] 'unique and beautiful artistry seen on each horse' -> 'unique and beautiful artistry seen on each object'
  [REPLACED] 'overall form is a masterpiece of natural engineering for speed and endurance' -> 'form is a masterpiece of natural engineering for speed and endurance'
  [REPLACED] 'overall impression was one of robust health and understated elegance' -> 'impression was one of robust health and understated elegance'
  [REPLACED] 'overall impression is one of classic elegance and a well-defined co

In [21]:
# get me count of extracted concepts per class after cleaning
for dataset in os.listdir('llm_outputs'):
    file_path = os.path.join('llm_outputs', dataset, "data.json")
    with open(file_path, "r") as f:
        data_json = json.load(f)

    print(f"\nConcept counts for {dataset}:")
    for class_name, content in data_json[dataset].items():
        concepts = content.get("concepts", [])
        print(f"  {class_name}: {len(concepts)} concepts")


Concept counts for CIFAR10:
  airplane: 751 concepts
  automobile: 1009 concepts
  bird: 727 concepts
  cat: 881 concepts
  deer: 837 concepts
  dog: 859 concepts
  frog: 906 concepts
  horse: 967 concepts
  ship: 803 concepts
  truck: 762 concepts


In [25]:
# get me count of extracted concepts per class after cleaning
new = {}
for dataset in os.listdir('llm_outputs'):
    file_path = os.path.join('llm_outputs', dataset, "data.json")
    with open(file_path, "r") as f:
        data_json = json.load(f)

    print(f"\nConcept counts for {dataset}:")
    for class_name, content in data_json[dataset].items():
        concepts = content.get("concepts", [])
        new[class_name] = concepts


Concept counts for CIFAR10:


In [26]:
with open("datasets/CIFAR10/concepts/class2concepts_gemini.json", "w") as f:
    json.dump(new, f, indent=4)

In [30]:
from transformers import pipeline
generator =  pipeline("mask-generation", device = 0, points_per_batch = 256)
image_url = "https://huggingface.co/ybelkada/segment-anything/resolve/main/assets/car.png"
outputs = generator(image_url, points_per_batch = 256)


No model was supplied, defaulted to facebook/sam-vit-huge and revision 87aecf0 (https://huggingface.co/facebook/sam-vit-huge).
Using a pipeline without specifying a model name and revision in production is not recommended.
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Device set to use cuda:0


OutOfMemoryError: CUDA out of memory. Tried to allocate 13.36 GiB. GPU 0 has a total capacity of 22.04 GiB of which 11.44 GiB is free. Process 3110151 has 478.00 MiB memory in use. Process 156948 has 280.00 MiB memory in use. Process 179295 has 266.00 MiB memory in use. Process 179470 has 266.00 MiB memory in use. Process 179132 has 266.00 MiB memory in use. Process 179219 has 266.00 MiB memory in use. Process 179185 has 266.00 MiB memory in use. Process 179294 has 266.00 MiB memory in use. Process 179088 has 266.00 MiB memory in use. Process 179291 has 266.00 MiB memory in use. Process 179287 has 266.00 MiB memory in use. Process 179381 has 266.00 MiB memory in use. Including non-PyTorch memory, this process has 7.21 GiB memory in use. Of the allocated memory 6.56 GiB is allocated by PyTorch, and 434.68 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See docume

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

def show_mask(mask, ax, random_color=False):
    if random_color:
        color = np.concatenate([np.random.random(3), np.array([0.6])], axis=0)
    else:
        color = np.array([30 / 255, 144 / 255, 255 / 255, 0.6])
    h, w = mask.shape[-2:]
    mask_image = mask.reshape(h, w, 1) * color.reshape(1, 1, -1)
    ax.imshow(mask_image)
    

plt.imshow(np.array(raw_image))
ax = plt.gca()
for mask in outputs["masks"]:
    show_mask(mask, ax=ax, random_color=True)
plt.axis("off")
plt.show()


In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "7"

In [6]:
import json

with open('llm_out/CIFAR10/data.json', 'r') as f:
    data = json.load(f)



with open('datasets/CIFAR10/concepts/class2concepts_gemini.json', 'w') as f:
    json.dump(data['CIFAR10'], f, indent=4)

In [106]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

test_img_feat = torch.load('./datasets/flower/splits/img_feat_test_00_ViT-L-14.pth').to(device)

device, test_img_feat.shape

('cuda', torch.Size([2463, 768]))

In [82]:
import numpy as np
cls_names = np.load("datasets/CIFAR10/concepts/cls_names.npy", allow_pickle=True)
print(list(cls_names))

['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


In [81]:
import pickle

with open("datasets/CIFAR10/splits/class2images_train.p", "rb") as f:
    d = pickle.load(f)

print(list(d.keys()))

['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


In [83]:
import numpy as np
import pickle

cls_names = np.load("datasets/CIFAR10/concepts/cls_names.npy", allow_pickle=True).tolist()

with open("datasets/CIFAR10/splits/class2images_train.p", "rb") as f:
    split_keys = list(pickle.load(f).keys())

print("cls_names.npy:")
for i, name in enumerate(cls_names):
    print(i, name)

print("\nsplit keys:")
for i, name in enumerate(split_keys):
    print(i, name)

print("\nMissing from cls_names:", [x for x in split_keys if x not in cls_names])
print("Missing from split keys:", [x for x in cls_names if x not in split_keys])

cls_names.npy:
0 airplane
1 automobile
2 bird
3 cat
4 deer
5 dog
6 frog
7 horse
8 ship
9 truck

split keys:
0 airplane
1 automobile
2 bird
3 cat
4 deer
5 dog
6 frog
7 horse
8 ship
9 truck

Missing from cls_names: []
Missing from split keys: []


In [109]:
# find the top 5 concepts for this image
topk = 10
topk_indices = torch.topk(out, k=topk).indices.cpu().tolist()
print("Top concepts for test image 1000:")
for idx in topk_indices:
    print(f"  {idx}: {concept_texts[idx][:50]} score={out[idx].item():.4f}")

Top concepts for test image 1000:
  1748: adorned with oval or lance-shaped leaves that are  score=72.3862
  420: a hue reminiscent of polished rubies score=69.7981
  2052: almost like spun sunlight score=68.3222
  2095: also common score=67.4344
  2364: appear delicate and somewhat waxy score=67.2545
  99: a cluster of prominent, often reddish or yellow, s score=66.8986
  1371: adding a contrasting focal point score=66.4584
  938: accentuates the bolder outer colors score=66.0302
  2130: also possess a delicate, sweet fragrance score=65.7890
  1308: add to its unique structure score=65.3820
